* Issue [#300](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/300)
* [this Notebook](https://github.com/salgo60/Stockholm_Archipelago_Trail/blob/main/Notebook/SAT300_FilmingLocations.ipynb)

* [SAT Dashboard](https://salgo60.github.io/Stockholm_Archipelago_Trail/Notebook/output/dashboard.html)
   * karta [SAT300_FilmingLocations.html](https://salgo60.github.io/Stockholm_Archipelago_Trail/Notebook/output/SAT300_FilmingLocations.html)
   * 
    

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-08 17:17


In [6]:
from string import Template
from datetime import datetime as dt
import html  # vi använder html.escape här

def add_about_box(
    m,
    issue_number: int,
    map_name: str,
    created_date: str | None = None,
    repo: str = "salgo60/Stockholm_Archipelago_Trail",
    collapsed: bool = False,
    offset_px=(10, 54),  # (top, left) – 54px för att hamna snällt bredvid zoom
):
    """Superenkel, robust About-box som alltid syns."""
    if created_date is None:
        created_date = dt.now().strftime("%Y-%m-%d %H:%M")

    map_dom_id = m.get_name()
    box_id     = f"sat-about-{map_dom_id}"
    header_id  = f"{box_id}-hdr"
    issue_url  = f"https://github.com/{repo}/issues/{issue_number}"
    top, left  = offset_px
    collapsed_class = "sat-about-collapsed" if collapsed else ""

    links = [
        ("SAT Dashboard", "dashboard.html"),
        ("Project repo issues", "https://github.com/salgo60/Stockholm_Archipelago_Trail/issues?q=is%3Aissue"),
        ("Trail on OSM (rel 19012437)", "https://www.openstreetmap.org/relation/19012437"),
        ("Trail on Wikicommons", "https://commons.wikimedia.org/wiki/Category:Stockholm_Archipelago_Trail"),
        ("Official page", "https://stockholmarchipelagotrail.com/"),
        ("Unofficial FB group", "https://www.facebook.com/groups/2875020699552247"),
        ("Visit Sweden", "https://traveltrade.visitsweden.com/plan/news-sweden/Stockholm-Archipelago-Trail/"),
    ]
    links_html = "".join(
        f'<div><a href="{html.escape(u)}" target="_blank" style="text-decoration:none;">🔗 {html.escape(t)}</a></div>'
        for t, u in links
    )

    tpl = Template(r"""
<style>
  .sat-about {
    position: fixed; z-index: 10000;
    top: ${top}px; left: ${left}px;
    background: rgba(255,255,255,0.97);
    border: 2px solid #666; border-radius: 10px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
    font: 12px/1.35 -apple-system, system-ui, Segoe UI, Roboto, Helvetica, Arial, sans-serif;
    min-width: 240px; max-width: 320px; pointer-events: auto;
  }
  .sat-about-header { cursor: pointer; padding: 8px 10px; font-weight: 700;
    display: flex; align-items: center; gap: 6px; user-select: none;
    background: rgba(248,248,248,.9); border-bottom: 1px solid #e5e7eb; }
  .sat-about-body { padding: 8px 10px 10px 10px; }
  .sat-about-collapsed .sat-about-body { display: none; }
  .sat-chevron { margin-left: auto; transition: transform .15s ease-in-out; }
  .sat-about-collapsed .sat-chevron { transform: rotate(-90deg); }
  .sat-links { margin-top: 6px; padding-top: 6px; border-top: 1px solid #e5e7eb; }
</style>

<div id="${box_id}" class="sat-about ${collapsed_class}">
  <div id="${header_id}" class="sat-about-header" title="Click to collapse/expand">
    <span>ℹ️ About</span><span class="sat-chevron">▸</span>
  </div>
  <div class="sat-about-body">
    <div style="font-weight:700;margin-bottom:4px;">Stockholm Archipelago Trail Map</div>
    <div>Issue: <a href="${issue_url}" target="_blank">#${issue_number}</a>&nbsp;&nbsp; Map: ${map_name}</div>
    <div>Created: ${created_date}</div>
    <div>Latest updates: saved as <i>_latest.html</i></div>
    <div class="sat-links">${links_html}</div>
  </div>
</div>

<script>
(function(){
  var boxId = "${box_id}";
  var hdrId = "${header_id}";
  var storageKey = "satAboutCollapsed_${map_dom_id}_#${issue_number}";

  function setCollapsed(box, collapsed) {
    if (!box) return;
    if (collapsed) box.classList.add("sat-about-collapsed");
    else box.classList.remove("sat-about-collapsed");
    try { localStorage.setItem(storageKey, collapsed ? "1" : "0"); } catch(e) {}
  }

  function init(){
    var box = document.getElementById(boxId);
    var hdr = document.getElementById(hdrId);
    if (!box || !hdr) return;

    try {
      var stored = localStorage.getItem(storageKey);
      if (stored === "1") setCollapsed(box, true);
      if (stored === "0") setCollapsed(box, false);
    } catch(e) {}

    hdr.addEventListener("click", function(e){
      e.stopPropagation();
      setCollapsed(box, !box.classList.contains("sat-about-collapsed"));
    });
  }

  if (document.readyState === "loading") {
    document.addEventListener("DOMContentLoaded", init);
  } else {
    init();
  }
})();
</script>
""")

    html_code = tpl.substitute(
        box_id=box_id,
        header_id=header_id,
        issue_number=issue_number,
        issue_url=issue_url,
        map_name=html.escape(map_name),
        created_date=created_date,
        links_html=links_html,
        collapsed_class=collapsed_class,
        map_dom_id=map_dom_id,
        top=top, left=left,
    )
    m.get_root().html.add_child(folium.Element(html_code))


In [9]:
import folium
import requests
import json
from SPARQLWrapper import SPARQLWrapper, JSON  
from folium import plugins
import geopandas as gpd
import html


# --- Läs in SAT trailen ---
sat_geojson_path = "SAT_full.geojson"

# --- Fördefinierade inspelningsplatser med extra info ---
filmingLocation = [
    { "id": "wd:Q3378260", "plats": "Huvudskärs fyr", "desc": "Huvudskärs fyr där kon i avsnitt 1 gick vilse. Syns i avsnitt 1 vid 17.40",
      "avsnitt": "1. Nu börjar det", "avsnittref": "https://www.svtplay.se/video/jMd2ak5/vi-pa-saltkrakan-1/1-nu-borjar-det?video=visa&position=1063"},
    { "id": "wd:Q10684506", "plats": "Svartlöga", "desc": "Svartlöga där spelas många av scenerna in. Flygfotot i avsnitt 1 vid 1:30 är nog ön med en fyr infogad",
      "avsnitt": "1. Nu börjar det", "avsnittref": "https://www.svtplay.se/video/jMd2ak5/vi-pa-saltkrakan-1/1-nu-borjar-det?video=visa&position=90"},
    { "id": "wd:Q134523091", "plats": "Långviks brygga", "desc": "Långviks brygga dit båten ankommer och där Westerhamn får hjärtattack i sista avsnittet vid 19:00",
      "avsnitt": "6. Är det slut nu?", "avsnittref": "https://www.svtplay.se/video/eWmX4Pm/vi-pa-saltkrakan-1/6-ar-det-slut-nu?video=visa&position=1138"},
    { "id": "wd:Q135890833", "plats": "Långvik Coop", "desc": "Grankvists livs som drivs av Tjorvens föräldrar syns i avsnitt 1 vid 1:40",
      "avsnitt": "1. Nu börjar det", "avsnittref": "https://www.svtplay.se/video/jMd2ak5/vi-pa-saltkrakan-1/1-nu-borjar-det?position=100"},
    { "id": "wd:Q136687327", "plats": "Fru Sjöbloms hus Snickargården", "desc": "Huset Melker hyrt av Westerman. Syns i avsnitt 1 vid 5:30",
      "avsnitt": "1. Nu börjar det", "avsnittref": "https://www.svtplay.se/video/jMd2ak5/vi-pa-saltkrakan-1/1-nu-borjar-det?video=visa&position=327"},
    { "id": "wd:Q136687404", "plats": "Löka brygga", "desc": "När Tjorven sitter i regnet och väntar på bäten i avsnitt 1 vid 3:05",
      "avsnitt": "1. Nu börjar det", "avsnittref": "https://www.svtplay.se/video/jMd2ak5/vi-pa-saltkrakan-1/1-nu-borjar-det?video=visa&position=185"},
    { "id": "wd:Q139572724", "plats": "Svartlöga sjöbodarna", "desc": "Skrållan och Tjorven skall till Snickargården avsnitt 2 vid 7:20",
      "avsnitt": "2. Vilse i dimman", "avsnittref": "https://www.svtplay.se/video/eorzRkV/vi-pa-saltkrakan-1/2-vilse-i-dimman?video=visa&position=426"}
]

# --- Skapa en lista av Wikidata-ID utan prefix ---
wd_ids = [f["id"].replace("wd:", "") for f in filmingLocation]

# --- Hämta koordinater och bilder från Wikidata ---
query = f"""
SELECT ?place ?placeLabel ?coord ?image ?commonscat ?svwiki ?enwiki WHERE {{
  VALUES ?place {{ {' '.join('wd:' + i for i in wd_ids)} }}
  OPTIONAL {{ ?place wdt:P625 ?coord. }}
  OPTIONAL {{ ?place wdt:P18 ?image. }}
  OPTIONAL {{ ?place wdt:P373 ?commonscat. }}
  OPTIONAL {{ ?svwiki schema:about ?place ;
                    schema:isPartOf <https://sv.wikipedia.org/>. }}
  OPTIONAL {{ ?enwiki schema:about ?place ;
                    schema:isPartOf <https://en.wikipedia.org/>. }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "sv,en". }}
}}
"""

print("Time sleep 150sek")
time.sleep(150)

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

# --- Mappa resultatet till dictionary ---
wikidata_info = {}
for r in results["results"]["bindings"]:
    qid = r["place"]["value"].split("/")[-1]
    wikidata_info[qid] = {
        "label": r.get("placeLabel", {}).get("value", ""),
        "coord": r.get("coord", {}).get("value", None),
        "image": r.get("image", {}).get("value", None),
        "commonscat": r.get("commonscat", {}).get("value", None),
        "svwiki": r.get("svwiki", {}).get("value", None),
        "enwiki": r.get("enwiki", {}).get("value", None)
    }

# --- Skapa karta ---
m = folium.Map(location=[59.4, 18.8], zoom_start=8, tiles="OpenStreetMap")

# --- Lägg till lager på huvudkartan direkt (utan .add_to() tidigare) ---
osm_layer = folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap (karta)",
    attr="© OpenStreetMap contributors"
)
esri_layer = folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Imagery (satellit)",
    attr="© Esri, Maxar, Earthstar Geographics"
)

# --- Lägg till SAT trail ---
sat_layer = folium.FeatureGroup(name="SAT-spår", show=True)

folium.GeoJson(
    sat_geojson_path,
    style_function=lambda feature: {
        "color": "#ff0000",
        "weight": 3,
        "opacity": 0.8
    },
    name="Stockholm Archipelago Trail",
    tooltip="Stockholm Archipelago Trail"
).add_to(sat_layer)

filminglocations_layer = folium.FeatureGroup(name="Inspelningsplatser", show=True)

# --- Lägg till inspelningsplatser ---
for f in filmingLocation:
    qid = f["id"].replace("wd:", "")
    info = wikidata_info.get(qid, {})
    
    if not info.get("coord"):
        continue

    # Extrahera koordinater
    lat, lon = map(float, info["coord"].replace("Point(", "").replace(")", "").split()[::-1])

    # Bild
    image_html = ""
    if info.get("image"):
        image_url = info["image"].replace("http://commons.wikimedia.org/wiki/Special:FilePath/", "https://commons.wikimedia.org/wiki/File:")
        thumb_url = info["image"]
        image_html = f'<a href="{image_url}" target="_blank"><img src="{thumb_url}" width="250"></a><br>'

    # Commons-kategori
    commons_link = ""
    if info.get("commonscat"):
        commons_link = f'<a href="https://commons.wikimedia.org/wiki/Category:{info["commonscat"]}" target="_blank">📸 Wikimedia Commons kategori</a><br>'

    # Wikidata 
    wd_link = f"""
    <div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:8px;">
      <a href="https://w.wiki/FuJX" target="_blank" rel="noopener noreferrer"
         style="display:inline-flex;align-items:center;gap:6px;padding:6px 12px;
                border-radius:9999px;border:1px solid #d1d5db;
                background:#f9fafb;text-decoration:none;font-weight:600;font-size:14px;">
        🧠 Kunskapsgraf – Vi på Saltkråkan 2025
      </a>
      <a href="https://w.wiki/EMLz" target="_blank" rel="noopener noreferrer"
         style="display:inline-flex;align-items:center;gap:6px;padding:6px 12px;
                border-radius:9999px;border:1px solid #d1d5db;
                background:#f9fafb;text-decoration:none;font-weight:600;font-size:14px;">
        🧠 Kunskapsgraf – Stockholm Archipelago Trail
      </a>
    </div>
    """
    
    # SVT-avsnitt
    svt_link = f'SVT Play <a href="{f["avsnittref"]}" target="_blank">🎬 {f["avsnitt"]}</a>'
    svt_link = f'''
    <a href="{f["avsnittref"]}" target="_blank" rel="noopener noreferrer"
       style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;
              border-radius:9999px;border:1px solid #e2e8f0;background:#f8fafc;
              text-decoration:none;font-weight:600;">
      <span style="font-size:18px;line-height:1">🎬</span>
      <span>Se på SVT Play</span>
    </a>
    <div style="margin:2px 0 0 34px;font-size:14px;color:#334155;">
      {html.escape(f["avsnitt"])}
    </div>
    '''
    # Wikipedia-länkar
    wiki_links = ""
    if info.get("svwiki"):
        wiki_links += f'<a href="{info["svwiki"]}" target="_blank">🇸🇪 Wikipedia (svenska)</a><br>'
    if info.get("enwiki"):
        wiki_links += f'<a href="{info["enwiki"]}" target="_blank">🇬🇧 Wikipedia (English)</a><br>'


    # Extra länkar (kartor / appar)
    osm_link = f'<a href="https://osmand.net/go?lat={lat}&lon={lon}&z=17" target="_blank">📍 OsmAnd</a>'
    mapillary_link = f'<a href="https://www.mapillary.com/app/?lat={lat}&lng={lon}&z=17" target="_blank">🟢 Mapillary</a>'
    gmaps_link = f'<a href="https://www.google.com/maps?q={lat},{lon}" target="_blank">🗺️ Google Maps</a>'
    street_link = f'<a href="https://www.google.com/maps?q=&layer=c&cbll={lat},{lon}" target="_blank">📷 Street View</a>'

    popup_html = f"""
    <b>{f['plats']}</b><br>
    {image_html}
    <i>{f['desc']}</i><br><br>
    {commons_link}
    {svt_link}<br>
    {wiki_links}
    {wd_link}<br><br>
    {osm_link}<br>
    {mapillary_link}<br>
    {gmaps_link}<br>
    {street_link}
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f["plats"],
        icon=folium.Icon(color="red", icon="film")
    ).add_to(filminglocations_layer)

add_about_box(m, issue_number=300, map_name="Vi på Saltkråkan 2025") 

# --- Lägg till lager på huvudkartan direkt ---
osm_layer.add_to(m)
esri_layer.add_to(m)

# --- Lägg till SAT- och inspelningslager efteråt ---
sat_layer.add_to(m)
filminglocations_layer.add_to(m)


# --- Side-by-side kontroll (vänster = satellit, höger = karta) ---
plugins.SideBySideLayers(
    esri_layer,
    osm_layer
).add_to(m)

# --- Lagerkontroll ---
folium.LayerControl(position="topright", collapsed=False).add_to(m)

# --- Visa karta ---
display(m)

Time sleep 150sek


HTTPError: HTTP Error 429: Aggressively rate-limiting to 1 req / min - this rule was created during active wdqs outage (797a132)

In [ ]:
m.save("output/SAT300_FilmingLocations.html")

In [ ]:
wikidata_info

In [ ]:
end_time = time.time()
duration = end_time - start_time
print(f"Finished in {duration:.2f} seconds.")
